# Experiment 2: MongoDB Installation and CRUD Operations

## 1. Install MongoDB 7.0 and start the service

In [ ]:
%%bash
sudo apt install -y gnupg curl
curl -fsSL https://www.mongodb.org/static/pgp/server-7.0.asc | \
  sudo gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor
echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | \
  sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list
sudo apt update
sudo apt install -y mongodb-org
sudo systemctl start mongod
sudo systemctl enable mongod
sudo systemctl status mongod     # should show active (running)

## 2. Open the shell
Run `mongosh` in a terminal. The cells below use `%%script mongosh`, so each cell starts a fresh shell session (hence `use college` at the top of each).

## 3. Database + collection

In [ ]:
%%script mongosh
show dbs
use college
db
db.createCollection("students")
show collections

## 4. CREATE

In [ ]:
%%script mongosh
use college
db.students.insertOne({ roll: 1, name: "Rahul", dept: "AIDS", marks: 85 })
db.students.insertMany([
  { roll: 2, name: "Amit",  dept: "IT",   marks: 90 },
  { roll: 3, name: "Sneha", dept: "AIDS", marks: 78 },
  { roll: 4, name: "Priya", dept: "CS",   marks: 88 }
])

## 5. READ

In [ ]:
%%script mongosh
use college
db.students.find()
db.students.findOne({ roll: 3 })
db.students.find({ dept: "AIDS" })
db.students.find({ marks: { $gt: 80 } })
db.students.find({ marks: { $gte: 80, $lte: 90 } })
db.students.find({ marks: { $ne: 85 } })
db.students.find({ $or: [{ dept: "IT" }, { dept: "CS" }] })
db.students.find({ $and: [{ dept: "AIDS" }, { marks: { $gt: 80 } }] })
db.students.find({ dept: { $in: ["IT", "CS"] } })
db.students.find({ name: /^S/ })
db.students.find({ dept: { $exists: true } })
db.students.find({}, { name: 1, marks: 1, _id: 0 })
db.students.find().sort({ marks: -1 })
db.students.find().skip(1).limit(2)
db.students.countDocuments({ dept: "AIDS" })
db.students.distinct("dept")

## 6. UPDATE

In [ ]:
%%script mongosh
use college
db.students.updateOne({ roll: 1 }, { $set: { marks: 92 } })
db.students.updateMany({ dept: "AIDS" }, { $inc: { marks: 2 } })
db.students.updateOne({ roll: 2 }, { $rename: { marks: "score" } })
db.students.updateOne({ roll: 2 }, { $unset: { score: "" } })
db.students.updateOne({ roll: 3 }, { $push: { skills: "Python" } })
db.students.updateOne({ roll: 3 }, { $addToSet: { skills: "Python" } })
db.students.updateOne({ roll: 3 }, { $pull: { skills: "Python" } })
db.students.updateOne({ roll: 5 },
  { $set: { name: "Ravi", dept: "IT", marks: 70 } }, { upsert: true })
db.students.replaceOne({ roll: 5 },
  { roll: 5, name: "Ravi K", dept: "CS", marks: 75 })

## 7. DELETE

In [ ]:
%%script mongosh
use college
db.students.deleteOne({ roll: 4 })
db.students.deleteMany({ marks: { $lt: 80 } })

## 8. INDEXES

In [ ]:
%%script mongosh
use college
db.students.createIndex({ roll: 1 })
db.students.getIndexes()
db.students.dropIndex({ roll: 1 })

## 9. AGGREGATION

In [ ]:
%%script mongosh
use college
db.students.aggregate([
  { $group: { _id: "$dept", avgMarks: { $avg: "$marks" }, total: { $sum: 1 } } }
])
db.students.aggregate([
  { $match: { marks: { $gt: 80 } } },
  { $sort: { marks: -1 } },
  { $project: { _id: 0, name: 1, marks: 1 } }
])

## 10. CLEANUP

In [ ]:
%%script mongosh
use college
db.students.renameCollection("learners")
db.learners.drop()
db.dropDatabase()